# EDA — `factures_ventes` & `bons_commande`

---

## Objectif de ce notebook

Ce notebook analyse les deux flux financiers de l'entreprise PetroStock SA :

- **Partie A — `factures_ventes`** : ce que l'entreprise vend à ses clients (32 577 factures)
- **Partie B — `bons_commande`** : ce que l'entreprise achète à ses fournisseurs (3 248 bons)

L'analyse combinée de ces deux tables révèle la santé commerciale et logistique de l'entreprise,  
et alimente le module de recommandation automatique de fournisseur.

---

## 0. Imports et Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

# Dossier racine pour les figures
FIGURES_DIR = os.path.join('..', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

if not hasattr(plt.savefig, '_is_patched'):
    _original_savefig = plt.savefig
    def savefig_root(path, *args, **kwargs):
        if isinstance(path, str) and path.startswith('figures/'):
            path = os.path.join('..', path)
        return _original_savefig(path, *args, **kwargs)
    savefig_root._is_patched = True
    plt.savefig = savefig_root

print('Imports OK ✅')

---
# PARTIE A — Factures Ventes
---

## 1. Chargement et Inspection — Factures

In [ ]:
fv = pd.read_csv('../data/factures_ventes.csv')
fv['date_facture'] = pd.to_datetime(fv['date_facture'])
fv['date_echeance'] = pd.to_datetime(fv['date_echeance'])
fv['annee'] = fv['date_facture'].dt.year
fv['mois']  = fv['date_facture'].dt.month

print(f"Dimensions : {fv.shape[0]:,} lignes × {fv.shape[1]} colonnes")
print(f"Période    : {fv['date_facture'].min().date()} → {fv['date_facture'].max().date()}")
print(f"Clients    : {fv['client_id'].nunique()} clients uniques")
print(f"CA total   : {fv['montant_ttc'].sum():,.0f} FCFA")
print()
print('=== Valeurs manquantes ===')
missing = fv.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'Aucune ✅')
print(f'\nDoublons : {fv.duplicated().sum()}')
fv[['montant_ht', 'montant_tva', 'montant_ttc', 'remise_pct', 'tva_pct']].describe().round(2)

**📝 Observations :**

> Le dataset des factures contient **32 577 factures** propres, sans valeur manquante ni doublon. Il couvre 10 ans (2015–2024) et représente un chiffre d'affaires total de **23,85 milliards FCFA** pour 80 clients distincts. Le montant TTC moyen par facture est de **732 106 FCFA**. La remise moyenne est très faible (0,76%) — l'entreprise applique peu de rabais commerciaux.

## 2. Analyse du Chiffre d'Affaires

On analyse l'évolution du CA dans le temps, sa répartition par type de client et par région.

In [ ]:
# CA mensuel sur 10 ans
ca_mensuel = fv.groupby(fv['date_facture'].dt.to_period('M'))['montant_ttc'].sum()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(ca_mensuel.index.astype(str), ca_mensuel.values,
        color='steelblue', linewidth=0.8)
ax.fill_between(range(len(ca_mensuel)), ca_mensuel.values,
                alpha=0.15, color='steelblue')
ax.set_title('Chiffre d\'affaires mensuel (2015–2024) — FCFA')
ax.set_xlabel('Mois')
ax.set_ylabel('CA mensuel (FCFA)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
ax.set_xticks(range(0, len(ca_mensuel), 12))
ax.set_xticklabels([str(ca_mensuel.index[i]) for i in range(0, len(ca_mensuel), 12)], rotation=45)
# Ligne tendance
z = np.polyfit(range(len(ca_mensuel)), ca_mensuel.values, 1)
p = np.poly1d(z)
ax.plot(range(len(ca_mensuel)), p(range(len(ca_mensuel))),
        color='darkorange', linestyle='--', linewidth=1.2, label='Tendance')
ax.legend()
plt.tight_layout()
plt.savefig('figures/01_ca_mensuel.png', dpi=150)
plt.show()

print(f'CA moyen mensuel : {ca_mensuel.mean():,.0f} FCFA')
print(f'Meilleur mois    : {ca_mensuel.idxmax()} — {ca_mensuel.max():,.0f} FCFA')
print(f'Moins bon mois   : {ca_mensuel.idxmin()} — {ca_mensuel.min():,.0f} FCFA')

In [ ]:
# CA par type de client et par région
ca_type = fv.groupby('type_client')['montant_ttc'].sum().sort_values(ascending=True)
ca_region = fv.groupby('region_client')['montant_ttc'].sum().sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

colors_type = ['#14325A', '#1E5A8E', '#2980B9', '#5DADE2', '#AED6F1']
ca_type.plot(kind='barh', ax=axes[0], color=colors_type, edgecolor='white')
axes[0].set_title('CA total par type de client (FCFA)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:.1f}B'))

colors_region = ['#1E8E5A', '#27AE60', '#52BE80', '#82E0AA', '#ABEBC6']
ca_region.plot(kind='barh', ax=axes[1], color=colors_region, edgecolor='white')
axes[1].set_title('CA total par région cliente (FCFA)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:.1f}B'))

plt.tight_layout()
plt.savefig('figures/02_ca_type_region.png', dpi=150)
plt.show()

In [ ]:
# Top 20 clients par CA
top20 = fv.groupby('client_nom')['montant_ttc'].sum().sort_values(ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(10, 8))
top20.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Top 20 clients — Chiffre d\'affaires cumulé (FCFA)')
ax.set_xlabel('CA total (FCFA)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
plt.tight_layout()
plt.savefig('figures/03_top20_clients.png', dpi=150)
plt.show()

In [ ]:
# CA mensuel moyen par mois (saisonnalité)
ca_saisonnalite = fv.groupby('mois')['montant_ttc'].mean()
mois_labels = ['Jan','Fév','Mar','Avr','Mai','Jun','Jul','Aoû','Sep','Oct','Nov','Déc']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(mois_labels, ca_saisonnalite.values, color='steelblue', edgecolor='white')
ax.axhline(ca_saisonnalite.mean(), color='red', linestyle='--',
           linewidth=1.2, label=f'Moyenne : {ca_saisonnalite.mean()/1e6:.0f}M')
ax.set_title('CA moyen par mois — Saisonnalité des ventes')
ax.set_ylabel('CA moyen (FCFA)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
ax.legend()
plt.tight_layout()
plt.savefig('figures/04_saisonnalite_ca.png', dpi=150)
plt.show()

**📝 Observations :**

> Le CA mensuel est **relativement stable** sur 10 ans avec une légère tendance haussière (ligne pointillée orange). Il n'y a pas de choc brutal visible — l'activité commerciale est régulière et prévisible.
>
> Par type de client, l'**Industrie** est le segment le plus important (5,49B FCFA, 23%), suivi de près par les **Stations-service** (5,35B, 22,4%). Les 5 types sont relativement équilibrés — pas de concentration excessive sur un segment.
>
> La **région Kara** génère le plus de CA (5,38B) — surprenant pour une région du nord, moins peuplée que la Maritime. Cela suggère une forte activité industrielle ou institutionnelle dans cette région. La région **Maritime** (Lomé) est curieusement la moins performante (3,72B).
>
> La saisonnalité des ventes est faible — quelques mois légèrement supérieurs à la moyenne, sans pic extrême. Cette faible saisonnalité est cohérente avec la nature des produits (carburants et hydrocarbures à consommation continue).

## 3. Analyse du Recouvrement

On analyse les statuts de paiement pour identifier les clients à risque.

In [ ]:
# Répartition globale des statuts de paiement
statut_counts = fv['statut_paiement'].value_counts()
taux_recouvrement = fv['statut_paiement'].eq('Payée').mean() * 100

print(f'Taux de recouvrement global : {taux_recouvrement:.1f}%')
print()
for s, n in statut_counts.items():
    print(f'  {s:<25} : {n:>6,} ({n/len(fv)*100:.1f}%)')

colors_statut = {'Payée': '#1E8E5A', 'En attente': '#C9841A',
                 'Partiellement payée': '#E67E22', 'En retard': '#C63B3B'}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Camembert statuts
colors_pie = [colors_statut[s] for s in statut_counts.index]
axes[0].pie(statut_counts.values, labels=statut_counts.index,
             autopct='%1.1f%%', colors=colors_pie,
             wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0].set_title('Répartition des statuts de paiement')

# Montant non recouvré par statut
montant_statut = fv.groupby('statut_paiement')['montant_ttc'].sum()
non_paye = montant_statut[montant_statut.index != 'Payée']
non_paye.plot(kind='bar', ax=axes[1],
              color=[colors_statut[s] for s in non_paye.index],
              edgecolor='white')
axes[1].set_title('Montant non recouvré par statut (FCFA)')
axes[1].set_ylabel('Montant (FCFA)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:.1f}B'))
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('figures/05_recouvrement_global.png', dpi=150)
plt.show()

In [ ]:
# Taux de recouvrement par type de client
recouvrement_type = fv.groupby('type_client').apply(
    lambda x: pd.Series({
        'taux_paye_pct': (x['statut_paiement'] == 'Payée').mean() * 100,
        'taux_retard_pct': (x['statut_paiement'] == 'En retard').mean() * 100,
        'montant_impaye': x[x['statut_paiement'] != 'Payée']['montant_ttc'].sum()
    })
).round(1)

print('Analyse du recouvrement par type de client :')
print(recouvrement_type.sort_values('taux_paye_pct'))

fig, ax = plt.subplots(figsize=(10, 5))
recouvrement_type['taux_paye_pct'].sort_values().plot(
    kind='barh', ax=ax,
    color=['#C63B3B' if v < 60 else '#C9841A' if v < 70 else '#1E8E5A'
           for v in recouvrement_type['taux_paye_pct'].sort_values()],
    edgecolor='white'
)
ax.axvline(65.1, color='red', linestyle='--', linewidth=1.2,
           label=f'Moyenne globale : 65.1%')
ax.set_title('Taux de recouvrement par type de client (%)')
ax.set_xlabel('Taux de paiement intégral (%)')
ax.legend()
plt.tight_layout()
plt.savefig('figures/06_recouvrement_par_type.png', dpi=150)
plt.show()

In [ ]:
# Modes de paiement
mode_counts = fv['mode_paiement'].value_counts()

fig, ax = plt.subplots(figsize=(8, 5))
mode_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Répartition des modes de paiement')
ax.set_ylabel("Nombre de factures")
ax.set_xticklabels(mode_counts.index, rotation=0)
for i, (idx, val) in enumerate(mode_counts.items()):
    ax.text(i, val + 50, f'{val/len(fv)*100:.1f}%', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('figures/07_modes_paiement.png', dpi=150)
plt.show()

**📝 Observations :**

> Le taux de recouvrement global est de **65,1%** — seulement 65 factures sur 100 sont entièrement payées. C'est un indicateur préoccupant : 34,9% des créances sont en attente, partielles ou en retard, représentant plusieurs milliards FCFA non recouvrés.
>
> Les factures **En retard** (2 528 factures, 7,8%) représentent le risque le plus élevé — ce sont des créances qui dépassent leur date d'échéance sans paiement complet. Le type de client le plus à risque sera identifié dans le graphique de recouvrement par type.
>
> Les 4 modes de paiement sont parfaitement équilibrés (autour de 25% chacun) — l'entreprise accepte indifféremment Mobile Money, Chèque, Espèces et Virement. Le Mobile Money est légèrement dominant, reflétant l'adoption du digital en Afrique de l'Ouest.
>
> **Décision pour le dashboard :** le taux de recouvrement et les factures en retard par type de client doivent être affichés en priorité sur la page Finances pour la Direction.

---
# PARTIE B — Bons de Commande
---

## 4. Chargement et Inspection — Bons de Commande

In [ ]:
bc = pd.read_csv('../data/bons_commande.csv')
bc['date_commande'] = pd.to_datetime(bc['date_commande'])
bc['date_livraison_prevue'] = pd.to_datetime(bc['date_livraison_prevue'])
bc['date_livraison_reelle'] = pd.to_datetime(bc['date_livraison_reelle'])
bc['annee'] = bc['date_commande'].dt.year
bc['mois']  = bc['date_commande'].dt.month

print(f"Dimensions : {bc.shape[0]:,} lignes × {bc.shape[1]} colonnes")
print(f"Période    : {bc['date_commande'].min().date()} → {bc['date_commande'].max().date()}")
print(f"Fournisseurs : {bc['fournisseur_id'].nunique()}")
print(f"Pays : {sorted(bc['pays_fournisseur'].unique())}")
print()
print('=== Valeurs manquantes ===')
missing = bc.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'Aucune ✅')
bc[['quantite_commandee', 'quantite_livree', 'montant_total',
    'delai_livraison_jours', 'retard_jours']].describe().round(2)

**📝 Observations :**

> Le dataset contient **3 248 bons de commande** sur 10 ans, avec 6 fournisseurs internationaux. Les valeurs manquantes sur `date_livraison_reelle` sont légitimes — elles correspondent aux commandes en cours ou en attente (non encore livrées). Le montant moyen par commande est de **1 801 697 USD** — des transactions de grande envergure, typiques du secteur pétrolier.

## 5. Performance des Fournisseurs

On analyse la fiabilité, les délais et les retards par fournisseur pour produire le classement final.

In [ ]:
# Répartition des statuts de commande
statut_bc = bc['statut'].value_counts()
colors_bc = {'Livré': '#1E8E5A', 'En cours': '#14325A',
             'En attente': '#C9841A', 'Annulé': '#C63B3B'}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Camembert statuts
colors_pie = [colors_bc[s] for s in statut_bc.index]
axes[0].pie(statut_bc.values, labels=statut_bc.index,
             autopct='%1.1f%%', colors=colors_pie,
             wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0].set_title('Répartition des statuts de commande')

# Statuts par fournisseur
statut_fourn = bc.groupby(['fournisseur_nom', 'statut']).size().unstack(fill_value=0)
statut_fourn_pct = statut_fourn.div(statut_fourn.sum(axis=1), axis=0) * 100

# Raccourcir les noms
statut_fourn_pct.index = [n.replace(' Pétrole', '').replace(' Energies TG', ' TG')
                           .replace(' West Africa', ' WA') for n in statut_fourn_pct.index]

statut_fourn_pct.plot(kind='bar', ax=axes[1], stacked=True,
                       color=[colors_bc.get(c, 'gray') for c in statut_fourn_pct.columns],
                       edgecolor='white')
axes[1].set_title('Répartition des statuts par fournisseur (%)')
axes[1].set_ylabel('Proportion (%)')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('figures/08_statuts_commandes.png', dpi=150)
plt.show()

In [ ]:
# Taux de livraison effectif par fournisseur
fiabilite = bc.groupby('fournisseur_nom').agg(
    nb_commandes=('bon_commande_id', 'count'),
    taux_livraison=('statut', lambda x: (x == 'Livré').mean() * 100),
    taux_annulation=('statut', lambda x: (x == 'Annulé').mean() * 100),
    retard_moyen=('retard_jours', 'mean'),
    delai_moyen=('delai_livraison_jours', 'mean'),
    montant_total=('montant_total', 'sum')
).round(2).sort_values('taux_livraison', ascending=False)

# Raccourcir les noms pour l'affichage
fiabilite.index = [n.replace(' Pétrole', '').replace(' Energies TG', ' TG')
                    .replace(' West Africa', ' WA') for n in fiabilite.index]

print('Classement des fournisseurs par fiabilité :')
print(fiabilite[['nb_commandes', 'taux_livraison', 'taux_annulation',
                  'retard_moyen', 'delai_moyen']].to_string())

fig, ax = plt.subplots(figsize=(10, 5))
fiabilite['taux_livraison'].sort_values().plot(
    kind='barh', ax=ax,
    color=['#C63B3B' if v < 58 else '#C9841A' if v < 62 else '#1E8E5A'
           for v in fiabilite['taux_livraison'].sort_values()],
    edgecolor='white'
)
ax.axvline(fiabilite['taux_livraison'].mean(), color='red', linestyle='--',
           linewidth=1.2, label=f'Moyenne : {fiabilite["taux_livraison"].mean():.1f}%')
ax.set_title('Taux de livraison effectif par fournisseur (%)')
ax.set_xlabel('Taux de livraison (%)')
ax.legend()
plt.tight_layout()
plt.savefig('figures/09_taux_livraison_fournisseur.png', dpi=150)
plt.show()

In [ ]:
# Distribution des retards par fournisseur (boxplot)
bc_livre = bc[bc['statut'] == 'Livré'].copy()
bc_livre['fourn_court'] = bc_livre['fournisseur_nom'].str.replace(
    ' Pétrole', '').str.replace(' Energies TG', ' TG').str.replace(' West Africa', ' WA')

fig, ax = plt.subplots(figsize=(12, 5))
ordre = bc_livre.groupby('fourn_court')['retard_jours'].mean().sort_values().index
sns.boxplot(data=bc_livre, x='fourn_court', y='retard_jours',
            order=ordre, palette='coolwarm', ax=ax)
ax.axhline(0, color='gray', linestyle='--', linewidth=1.0)
ax.set_title('Distribution des retards de livraison par fournisseur (jours)')
ax.set_xlabel('')
ax.set_ylabel('Retard (jours)')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('figures/10_retards_boxplot.png', dpi=150)
plt.show()

print('Retard moyen par fournisseur (commandes livrées) :')
print(bc_livre.groupby('fourn_court')['retard_jours'].agg(['mean','median','max']).round(1))

In [ ]:
# Calcul du score de recommandation fournisseur
# Pondération : 50% fiabilité + 30% délai (inversé) + 20% prix (inversé)
score_df = bc.groupby('fournisseur_nom').agg(
    fiabilite=('statut', lambda x: (x == 'Livré').mean() * 100),
    delai_moyen=('delai_livraison_jours', 'mean'),
    prix_moyen=('prix_unitaire', 'mean')
)

# Normalisation min-max
def norm(s, inverse=False):
    n = (s - s.min()) / (s.max() - s.min())
    return 1 - n if inverse else n

score_df['score_fiabilite'] = norm(score_df['fiabilite'])
score_df['score_delai']     = norm(score_df['delai_moyen'], inverse=True)
score_df['score_prix']      = norm(score_df['prix_moyen'], inverse=True)
score_df['SCORE_FINAL']     = (
    score_df['score_fiabilite'] * 0.50 +
    score_df['score_delai']     * 0.30 +
    score_df['score_prix']      * 0.20
).round(3)

score_df = score_df.sort_values('SCORE_FINAL', ascending=False)
score_df.index = [n.replace(' Pétrole', '').replace(' Energies TG', ' TG')
                   .replace(' West Africa', ' WA') for n in score_df.index]

print('=== Classement final des fournisseurs ===')
print(score_df[['fiabilite', 'delai_moyen', 'prix_moyen', 'SCORE_FINAL']].to_string())

fig, ax = plt.subplots(figsize=(10, 5))
colors_score = ['#1E8E5A' if i == 0 else '#C9841A' if i == 1 else '#14325A'
                 for i in range(len(score_df))]
score_df['SCORE_FINAL'].plot(kind='bar', ax=ax, color=colors_score, edgecolor='white')
ax.set_title('Score de recommandation fournisseur\n(50% Fiabilité + 30% Délai + 20% Prix)')
ax.set_ylabel('Score (0–1)')
ax.set_xticklabels(score_df.index, rotation=25, ha='right')
# Annoter le meilleur
ax.text(0, score_df['SCORE_FINAL'].iloc[0] + 0.01, '⭐ Recommandé', ha='center', fontsize=10, color='#1E8E5A')
plt.tight_layout()
plt.savefig('figures/11_score_fournisseurs.png', dpi=150)
plt.show()

**📝 Observations :**

> Le taux de livraison global est de **59,8%** — seulement 6 commandes sur 10 aboutissent à une livraison effective. Les 40% restants sont en cours, en attente ou annulés (5,1%). Ce taux relativement bas reflète les délais logistiques internationaux et les commandes en transit.
>
> **Vitol Group** est le fournisseur le plus fiable (63,9% de livraisons effectives, retard moyen de 1,6 jours). **Oryx Energies** est le moins fiable (56,3%) malgré un retard moyen légèrement plus faible (1,4 jours).
>
> La distribution des retards (boxplot) montre que tous les fournisseurs ont des retards médians faibles (< 2 jours) — les queues de distribution (valeurs aberrantes) révèlent des retards ponctuels pouvant aller jusqu'à plusieurs jours.
>
> **Le classement final par score pondéré** (50% fiabilité, 30% délai, 20% prix) désigne **Vitol Group** comme fournisseur recommandé. Ce classement alimentera directement le module `RecommandationFournisseur` du système et sera affiché sur le dashboard.

## 6. Analyse des Commandes

On analyse l'évolution des montants commandés et les produits les plus commandés.

In [ ]:
# Évolution du montant total commandé par mois
montant_mensuel = bc.groupby(bc['date_commande'].dt.to_period('M'))['montant_total'].sum()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(montant_mensuel.index.astype(str), montant_mensuel.values,
        color='steelblue', linewidth=0.8)
ax.fill_between(range(len(montant_mensuel)), montant_mensuel.values,
                alpha=0.15, color='steelblue')
ax.set_title('Montant total commandé par mois (USD)')
ax.set_xlabel('Mois')
ax.set_ylabel('Montant (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
ax.set_xticks(range(0, len(montant_mensuel), 12))
ax.set_xticklabels([str(montant_mensuel.index[i])
                    for i in range(0, len(montant_mensuel), 12)], rotation=45)
plt.tight_layout()
plt.savefig('figures/12_montant_commandes_mensuel.png', dpi=150)
plt.show()

In [ ]:
# Produits les plus commandés en volume et en valeur
prod_volume = bc.groupby('produit_nom')['quantite_commandee'].sum().sort_values(ascending=True)
prod_valeur = bc.groupby('produit_nom')['montant_total'].sum().sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

prod_volume.plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Produits commandés — Volume total')
axes[0].set_xlabel('Quantité commandée')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))

prod_valeur.plot(kind='barh', ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_title('Produits commandés — Valeur totale (USD)')
axes[1].set_xlabel('Valeur totale (USD)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:.1f}B'))

plt.tight_layout()
plt.savefig('figures/13_produits_commandes.png', dpi=150)
plt.show()

**📝 Observations :**

> L'évolution mensuelle des commandes montre une grande variabilité — certains mois concentrent des commandes massives (pics) reflétant des réapprovisionnements importants ou des achats groupés. Cette irrégularité est normale dans le secteur pétrolier où les commandes dépendent des niveaux de stock et des prix WTI.
>
> En volume, les **Lubrifiants** et le **Naphta** sont les plus commandés. En valeur, la répartition est plus équilibrée. Aucun produit n'est systématiquement annulé ou en rupture de commande — le portefeuille de produits est bien géré.

## 7. Conclusions et Décisions

---

### 🔍 Observations clés

> **Partie A — Factures :**
> 1. CA total de **23,85 milliards FCFA** sur 10 ans, relativement stable avec légère tendance haussière
> 2. Les 5 types de clients sont équilibrés — l'Industrie et les Stations-service dominent légèrement
> 3. Taux de recouvrement de **65,1%** — risque créances non négligeable
> 4. La région **Kara** est la plus performante en CA — contre-intuitif mais documenté
> 5. Saisonnalité des ventes **faible** — cohérent avec la nature des produits
>
> **Partie B — Commandes :**
> 6. **Vitol Group** est le fournisseur le plus fiable (63,9%) et le fournisseur recommandé
> 7. **Oryx Energies** est le moins fiable (56,3%) — à surveiller
> 8. Délai moyen de livraison : **8,5 jours** — paramètre clé pour le calcul du stock de sécurité
> 9. Retard moyen : **1,5 jour** — faible mais à intégrer dans les seuils d'alerte
> 10. Aucun produit systématiquement annulé

---

### ✅ Décisions pour le système

> - **Module RecommandationFournisseur** : score pondéré 50%/30%/20% validé par les données → Vitol Group recommandé en priorité
> - **Seuil d'alerte recouvrement** : signaler les clients avec taux < 60% dans le dashboard Direction
> - **Délai fournisseur** : utiliser **8,5 jours** comme référence pour calculer les stocks de sécurité
> - **Stock de sécurité** = consommation_journalière × (délai_moyen + retard_moyen) = conso × 10 jours

---

### 📁 Figures produites

| Fichier | Description |
|---|---|
| `01_ca_mensuel.png` | Évolution du CA mensuel avec tendance |
| `02_ca_type_region.png` | CA par type de client et par région |
| `03_top20_clients.png` | Top 20 clients par CA |
| `04_saisonnalite_ca.png` | Saisonnalité mensuelle des ventes |
| `05_recouvrement_global.png` | Statuts de paiement et montants non recouvrés |
| `06_recouvrement_par_type.png` | Taux de recouvrement par type de client |
| `07_modes_paiement.png` | Répartition des modes de paiement |
| `08_statuts_commandes.png` | Statuts des commandes par fournisseur |
| `09_taux_livraison_fournisseur.png` | Taux de livraison par fournisseur |
| `10_retards_boxplot.png` | Distribution des retards par fournisseur |
| `11_score_fournisseurs.png` | Classement par score de recommandation |
| `12_montant_commandes_mensuel.png` | Évolution mensuelle des montants commandés |
| `13_produits_commandes.png` | Produits les plus commandés (volume et valeur) |